# PyContrails Hello World

## Objective

This notebook is the first technical experiment for the robust contrail
routing project.

The objective is to reproduce a minimal end-to-end pycontrails example
and understand how atmospheric conditions and aircraft trajectories
are used to identify contrail-forming conditions.

## Project phase

Phase 0 — Setup and reproduction

In [4]:
import pycontrails

print(pycontrails.__version__)

0.63.3


## 1. Create a synthetic flight

For this initial experiment, we create a simple synthetic aircraft
trajectory. The aircraft travels from longitude 0° to 5° and latitude
50° to 52° while cruising at approximately 11 km altitude.

This is not intended to represent a real flight. It is simply a controlled
trajectory that lets us understand how pycontrails represents flight data.

In [3]:
import numpy as np
import pandas as pd

from pycontrails import Flight

In [5]:
longitude = np.linspace(0, 5, 20)
latitude = np.linspace(50, 52, 20)
altitude = np.full(20, 11_000)

time = pd.date_range(
    "2022-03-01 00:00:00",
    "2022-03-01 01:00:00",
    periods=20,
)

flight = Flight(
    longitude=longitude,
    latitude=latitude,
    altitude=altitude,
    time=time,
    flight_id="hello_world",
)

In [6]:
flight

Flight [4 keys x 20 length, 1 attributes]
	Keys: longitude, latitude, time, altitude
	Attributes:
	time                [2022-03-01 00:00:00, 2022-03-01 01:00:00]
	longitude           [0.0, 5.0]
	latitude            [50.0, 52.0]
	altitude            [11000.0, 11000.0]
	flight_id           hello_world

In [7]:
flight.data.keys()

dict_keys(['longitude', 'latitude', 'time', 'altitude'])

In [8]:
flight["latitude"]

array([50.        , 50.10526316, 50.21052632, 50.31578947, 50.42105263,
       50.52631579, 50.63157895, 50.73684211, 50.84210526, 50.94736842,
       51.05263158, 51.15789474, 51.26315789, 51.36842105, 51.47368421,
       51.57894737, 51.68421053, 51.78947368, 51.89473684, 52.        ])

In [9]:
flight.altitude_ft

array([36089.23884514, 36089.23884514, 36089.23884514, 36089.23884514,
       36089.23884514, 36089.23884514, 36089.23884514, 36089.23884514,
       36089.23884514, 36089.23884514, 36089.23884514, 36089.23884514,
       36089.23884514, 36089.23884514, 36089.23884514, 36089.23884514,
       36089.23884514, 36089.23884514, 36089.23884514, 36089.23884514])

## 2. Load atmospheric data

The aircraft trajectory alone is not sufficient to determine whether
persistent contrails are likely to form.

We therefore combine the flight trajectory with atmospheric conditions.
For this first experiment we use ERA5 reanalysis data through the
pycontrails ARCO ERA5 interface.

The atmospheric variables of interest are air temperature and specific
humidity, which are used by the PCR model to determine the intersection
of the Schmidt-Appleman criterion (SAC) and ice-supersaturated regions
(ISSR).

In [10]:
from pycontrails.datalib.ecmwf import ERA5ARCO

era5 = ERA5ARCO(
    time=("2022-03-01T00", "2022-03-01T01"),
    variables=["air_temperature", "specific_humidity"],
    pressure_levels=[200, 250, 300],
)

In [11]:
met = era5.open_metdataset()

In [13]:
met.data.data_vars

Data variables:
    air_temperature    (longitude, latitude, level, time) float32 25MB dask.array<chunksize=(1440, 721, 3, 1), meta=np.ndarray>
    specific_humidity  (longitude, latitude, level, time) float32 25MB dask.array<chunksize=(1440, 721, 3, 1), meta=np.ndarray>

In [14]:
met.data.sizes

Frozen({'longitude': 1440, 'latitude': 721, 'level': 3, 'time': 2})

## 3. Persistent Contrail Regions (PCR)

A persistent contrail requires two conditions:

1. The Schmidt–Appleman criterion (SAC) is satisfied.
2. The surrounding atmosphere is ice-supersaturated (ISSR).

The intersection of these conditions defines a Persistent Contrail Region
(PCR).

This experiment evaluates PCR along the synthetic flight trajectory.

In [15]:
from pycontrails.models.pcr import PCR

In [17]:
import inspect
from pycontrails.models.pcr import PCR

print(inspect.signature(PCR))

(met: 'MetDataset | None' = None, params: 'ModelParams | dict[str, Any] | None' = None, **params_kwargs: 'Any') -> 'None'


In [18]:
pcr = PCR(
    met=met,
)

In [19]:
print(pcr)

PCR model
	Persistent contrail regions
	Params: {'copy_source': True, 'interpolation_method': 'linear', 'interpolation_bounds_error': False, 'interpolation_fill_value': nan, 'interpolation_localize': False, 'interpolation_use_indices': False, 'interpolation_q_method': None, 'verify_met': True, 'downselect_met': True, 'met_longitude_buffer': (0.0, 0.0), 'met_latitude_buffer': (0.0, 0.0), 'met_level_buffer': (0.0, 0.0), 'met_time_buffer': (np.timedelta64(0,'h'), np.timedelta64(0,'h')), 'rhi_threshold': 1.0, 'humidity_scaling': None, 'engine_efficiency': 0.3, 'fuel': JetA(fuel_name='Jet A-1', q_fuel=43130000.0, hydrogen_content=13.8, ei_co2=3.159, ei_h2o=1.23, ei_so2=0.0012, ei_sulphates=2.4489795918367345e-05, ei_oc=1.9999999999999998e-05)}



In [20]:
print(pcr.params)

{'copy_source': True, 'interpolation_method': 'linear', 'interpolation_bounds_error': False, 'interpolation_fill_value': nan, 'interpolation_localize': False, 'interpolation_use_indices': False, 'interpolation_q_method': None, 'verify_met': True, 'downselect_met': True, 'met_longitude_buffer': (0.0, 0.0), 'met_latitude_buffer': (0.0, 0.0), 'met_level_buffer': (0.0, 0.0), 'met_time_buffer': (np.timedelta64(0,'h'), np.timedelta64(0,'h')), 'rhi_threshold': 1.0, 'humidity_scaling': None, 'engine_efficiency': 0.3, 'fuel': JetA(fuel_name='Jet A-1', q_fuel=43130000.0, hydrogen_content=13.8, ei_co2=3.159, ei_h2o=1.23, ei_so2=0.0012, ei_sulphates=2.4489795918367345e-05, ei_oc=1.9999999999999998e-05)}


## 4. Evaluate persistent contrail regions along the flight

The PCR model combines the aircraft trajectory with the atmospheric
conditions and evaluates whether persistent contrail formation is
possible along the flight path.

In [27]:
pcr_flight = pcr.eval(source=flight)

In [23]:
pcr_flight

Flight [15 keys x 20 length, 6 attributes]
	Keys: longitude, latitude, time, altitude, flight_id, ..., pcr
	Attributes:
	time                [2022-03-01 00:00:00, 2022-03-01 01:00:00]
	longitude           [0.0, 5.0]
	latitude            [50.0, 52.0]
	altitude            [11000.0, 11000.0]
	flight_id           hello_world
	met_source_provider ECMWF
	met_source_dataset  ERA5
	met_source_product  reanalysis
	pycontrails_version 0.63.3
	engine_efficiency   0.3

In [24]:
pcr_flight.data.keys()

dict_keys(['longitude', 'latitude', 'time', 'altitude', 'flight_id', 'air_pressure', 'air_temperature', 'specific_humidity', 'issr', 'G', 'T_sat_liquid', 'rh', 'rh_critical_sac', 'sac', 'pcr'])

In [25]:
pcr_flight.data

{'longitude': array([0.        , 0.26315789, 0.52631579, 0.78947368, 1.05263158,
        1.31578947, 1.57894737, 1.84210526, 2.10526316, 2.36842105,
        2.63157895, 2.89473684, 3.15789474, 3.42105263, 3.68421053,
        3.94736842, 4.21052632, 4.47368421, 4.73684211, 5.        ]),
 'latitude': array([50.        , 50.10526316, 50.21052632, 50.31578947, 50.42105263,
        50.52631579, 50.63157895, 50.73684211, 50.84210526, 50.94736842,
        51.05263158, 51.15789474, 51.26315789, 51.36842105, 51.47368421,
        51.57894737, 51.68421053, 51.78947368, 51.89473684, 52.        ]),
 'time': array(['2022-03-01T00:00:00.000000000', '2022-03-01T00:03:09.473684000',
        '2022-03-01T00:06:18.947368000', '2022-03-01T00:09:28.421052000',
        '2022-03-01T00:12:37.894736000', '2022-03-01T00:15:47.368421000',
        '2022-03-01T00:18:56.842105000', '2022-03-01T00:22:06.315789000',
        '2022-03-01T00:25:15.789473000', '2022-03-01T00:28:25.263157000',
        '2022-03-01T00:31:34.

In [28]:
type(pcr_flight)

pycontrails.core.flight.Flight

In [33]:
results = pd.DataFrame({
    "time": pcr_flight.data["time"],
    "sac": pcr_flight.data["sac"],
    "issr": pcr_flight.data["issr"],
    "pcr": pcr_flight.data["pcr"],
})

results

,time,sac,issr,pcr
0,2022-03-01 00:00:00.000000,1.0,0.0,0.0
1,2022-03-01 00:03:09.473684,1.0,0.0,0.0
2,2022-03-01 00:06:18.947368,1.0,0.0,0.0
3,2022-03-01 00:09:28.421052,1.0,0.0,0.0
4,2022-03-01 00:12:37.894736,1.0,0.0,0.0
5,2022-03-01 00:15:47.368421,1.0,0.0,0.0
6,2022-03-01 00:18:56.842105,1.0,0.0,0.0
7,2022-03-01 00:22:06.315789,1.0,0.0,0.0
8,2022-03-01 00:25:15.789473,1.0,0.0,0.0
9,2022-03-01 00:28:25.263157,1.0,0.0,0.0


In [34]:
print("SAC:", pcr_flight.data["sac"].sum())
print("ISSR:", pcr_flight.data["issr"].sum())
print("PCR:", pcr_flight.data["pcr"].sum())

SAC: 20.0
ISSR: 0.0
PCR: 0.0


In [35]:
results = pd.DataFrame({
    "time": pcr_flight.data["time"],
    "temperature": pcr_flight.data["air_temperature"],
    "pressure": pcr_flight.data["air_pressure"],
    "RH": pcr_flight.data["rh"],
    "critical_RH": pcr_flight.data["rh_critical_sac"],
    "SAC": pcr_flight.data["sac"],
    "ISSR": pcr_flight.data["issr"],
    "PCR": pcr_flight.data["pcr"],
})

results

,time,temperature,pressure,RH,critical_RH,SAC,ISSR,PCR
0,2022-03-01 00:00:00.000000,213.216385,22631.70091,0.386965,0.0,1.0,0.0,0.0
1,2022-03-01 00:03:09.473684,213.042496,22631.70091,0.393818,0.0,1.0,0.0,0.0
2,2022-03-01 00:06:18.947368,212.874176,22631.70091,0.409457,0.0,1.0,0.0,0.0
3,2022-03-01 00:09:28.421052,212.760574,22631.70091,0.427309,0.0,1.0,0.0,0.0
4,2022-03-01 00:12:37.894736,212.662323,22631.70091,0.439702,0.0,1.0,0.0,0.0
5,2022-03-01 00:15:47.368421,212.570633,22631.70091,0.448597,0.0,1.0,0.0,0.0
6,2022-03-01 00:18:56.842105,212.426346,22631.70091,0.456876,0.0,1.0,0.0,0.0
7,2022-03-01 00:22:06.315789,212.264496,22631.70091,0.470448,0.0,1.0,0.0,0.0
8,2022-03-01 00:25:15.789473,212.094284,22631.70091,0.485195,0.0,1.0,0.0,0.0
9,2022-03-01 00:28:25.263157,211.969879,22631.70091,0.499295,0.0,1.0,0.0,0.0


In [36]:
print("Minimum RH:", results["RH"].min())
print("Maximum RH:", results["RH"].max())
print("Mean RH:", results["RH"].mean())

Minimum RH: 0.386965107970179
Maximum RH: 0.501083290788572
Mean RH: 0.44510076115468966
